Сверка: на основе одного файла саннотациями делаем второй

In [25]:
import csv

file1 = "all_cds_names_3.txt"
file2 = "1ab_2_final.csv"
outfile = "1ab_2_final_note.txt"

mapping = {}
with open(file2, encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == 2:
            key, value = row
            mapping[key.strip()] = value.strip()

with open(file1, encoding="utf-8") as f_in, open(outfile, "w", encoding="utf-8") as f_out:
    for line in f_in:
        word = line.strip()
        if word in mapping:
            f_out.write(f"{word},{mapping[word]}\n")
        else:
            f_out.write(word + "\n")


In [5]:
import csv

file1 = "1ab_2.txt"
file2 = "1b.csv"
outfile = "1ab_2_final.txt"

# читаем словарь из второго файла
mapping = {}
with open(file2, encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == 2:
            key, value = row
            mapping[key.strip()] = value.strip()

# обрабатываем первый файл
with open(file1, encoding="utf-8") as f_in, open(outfile, "w", encoding="utf-8") as f_out:
    for line in f_in:
        word = line.strip()
        # если в строке уже есть запятая → оставляем
        if "," in word:
            f_out.write(word + "\n")
        else:
            # если слово есть в словаре → добавляем через запятую
            if word in mapping:
                f_out.write(f"{word},{mapping[word]}\n")
            else:
                f_out.write(word + "\n")


Перевод из txt в csv

In [31]:
import csv

file_in = "1ab_2_final_note.txt"
file_out = "1ab_2_final_note.csv"

with open(file_in, encoding="utf-8") as f_in, open(file_out, "w", newline="", encoding="utf-8") as f_out:
    writer = csv.writer(f_out, delimiter=",")
    for line in f_in:
        line = line.strip()
        if not line:
            continue  # пропускаем пустые строки

        if "," in line:
            # делим только по последней запятой
            row = [x.strip() for x in line.rsplit(",", 1)]
        else:
            row = [line, ""]  # если нет запятой → вторая колонка пустая

        writer.writerow(row)


Вспомагательный скрипт для поиска в файле с расшифрованной аннотацией строк, где не написан продукт (нет условно ",1A")

In [28]:
import re

def check_lines(filename):
    """
    Проверяет строки в файле: каждая строка должна заканчиваться на ', ' или ', слово'
    """
    bad_lines = []
    pattern = re.compile(r",(\S.*)?$")  # строка должна заканчиваться ", " или ", слово"

    with open(filename, encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.rstrip("\n")
            if not pattern.search(line):
                bad_lines.append((i, line))

    return bad_lines


# запуск в юпитере
filename = input("Введите путь к файлу: ").strip()

bad_lines = check_lines(filename)

if bad_lines:
    print("\nНайдены строки, не соответствующие условию:")
    for i, text in bad_lines:
        print(f"Строка {i}: {text}")
else:
    print("Все строки соответствуют условию ✅")


Введите путь к файлу:  1ab_2_final_note.txt



Найдены строки, не соответствующие условию:
Строка 1: Number of records in file: 15566
Строка 2: Records with annotated CDS: 15144
Строка 3: CDS with product fields: 19705
Строка 4: CDS with gene fields: 4671
Строка 5: CDS with note fields: 7445
Строка 308: putative non-structural protein; ORF1b


Скрипт для вывода строк, где координаты в разных генах (1а, 1b, 1ab) совпадают

In [30]:
import pandas as pd

df = pd.read_csv("AV_2_modified_orf.txt", header=None)

# Функция проверки дубликатов координат в строке
def has_duplicates(row):
    # Берём все значения кроме первого столбца (id)
    coords = [x for x in row[1:] if x != "NA-NA"]
    return len(coords) != len(set(coords))

# Фильтруем строки с повторяющимися координатами
duplicates = df[df.apply(has_duplicates, axis=1)]

print(duplicates)


Empty DataFrame
Columns: [0, 1, 2, 3]
Index: []
